In [ ]:
# Cell 1: Setup and Data Loading
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Jupyter notebook magic command (most important fix)
%matplotlib inline

# ==================== PATH SETUP ====================
PROJECT_ROOT = Path.cwd().parent           # Goes from dashboard/ → Healthcare_AI_Project/
SRC_DIR = PROJECT_ROOT / "src"

print("Project Root:", PROJECT_ROOT)
print("SRC Dir:", SRC_DIR)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import CLUSTERED_CLAIMS_CSV, EXPLAINED_CLAIMS_CSV, OUTPUT_DIR, SCORED_CLAIMS_CSV
from plot_utils import (
    enrich_time_columns,
    plot_anomaly_spikes,
    plot_cluster_summary,
    plot_clusters,
    plot_provider_monthly_trend,
    run_kmeans,
)

OUTPUT_DIR.mkdir(exist_ok=True)

print("\nLoading dataset:")
print(SCORED_CLAIMS_CSV)
df = pd.read_csv(SCORED_CLAIMS_CSV)
print("Shape:", df.shape)

# Data preparation
required_cols = ["claim_amount", "Provider", "anomaly_flag", "risk_score"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Run train_model.py or score_pipeline.py first.")

df["anomaly_flag"] = pd.to_numeric(df["anomaly_flag"], errors="coerce").fillna(0).astype(int)

if "risk_category" not in df.columns:
    df["risk_category"] = np.where(df["anomaly_flag"] == 1, "high risk", "low risk")

plot_df = df.sample(n=min(10_000, len(df)), random_state=42)

print("Data ready for visualization.")

In [ ]:
# Cell 2: Enrich Time Columns and Run Clustering
print("Enriching time columns...")
df = enrich_time_columns(df)

print("\nRunning KMeans clustering...")
clustered_df, _, _ = run_kmeans(df)
clustered_df.to_csv(CLUSTERED_CLAIMS_CSV, index=False)
print("Saved:", CLUSTERED_CLAIMS_CSV)

In [ ]:
# Cell 3: Plot KMeans Clusters
fig, ax = plt.subplots(figsize=(8, 5))
plot_clusters(clustered_df, ax=ax)
fig.tight_layout()
path = OUTPUT_DIR / "kmeans_clusters.png"
fig.savefig(path, dpi=120)
plt.close()
print("Saved:", path)

In [ ]:
# Cell 4: Plot Cluster Summary
fig, ax = plt.subplots(figsize=(6, 4))
plot_cluster_summary(clustered_df, ax=ax)
fig.tight_layout()
path = OUTPUT_DIR / "cluster_summary.png"
fig.savefig(path, dpi=120)
plt.close()
print("Saved:", path)

In [ ]:
# Cell 5: Plot Provider Monthly Trend
fig, ax = plt.subplots(figsize=(9, 4))
plot_provider_monthly_trend(clustered_df, ax=ax)
fig.tight_layout()
path = OUTPUT_DIR / "provider_monthly_trend.png"
fig.savefig(path, dpi=120)
plt.close()
print("Saved:", path)

In [ ]:
# Cell 6: Plot Anomaly Spikes
fig, ax = plt.subplots(figsize=(9, 4))
plot_anomaly_spikes(clustered_df, ax=ax)
fig.tight_layout()
path = OUTPUT_DIR / "anomaly_spikes_over_time.png"
fig.savefig(path, dpi=120)
plt.close()
print("Saved:", path)

print("\nVisualization complete.")